# Walk-Forward + 실데이터 통합 테스트

**실행 환경**: Windows 로컬 + VS Code + Jupyter

**브랜치**: `claude/analyze-branch-files-EtGFl`

**순서**:
1. 환경 검증
2. Phase 단위 테스트 (Phase B~E)
3. 실데이터 로드 + 리스크 파이프라인 소표본 통합
4. Walk-Forward (Baseline + Enhanced)
5. Walk-Forward (Memory Enhanced, LLM 제한 실행)
6. 결과 분석 + 리포트 비교

## 1. 환경 검증

In [ ]:
import os, sys
# 작업 경로: 이 노트북 기준 상위 폴더로 이동
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(ROOT)
sys.path.insert(0, ROOT)
print(f"작업 폴더: {os.getcwd()}")

# API 키 로드 (dotenv)
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    print("  dotenv 미설치 (pip install python-dotenv)")

api_key = os.environ.get("ANTHROPIC_API_KEY", "")
print(f"API 키: {'OK (' + api_key[:15] + '...)' if api_key else '❌ 없음'}")

# 필수 패키지 확인
for pkg in ["pandas", "anthropic", "pydantic"]:
    try:
        __import__(pkg)
        print(f"  {pkg}: OK")
    except ImportError:
        print(f"  {pkg}: ❌ 설치 필요")

# 선택 패키지
for pkg in ["chromadb", "sentence_transformers", "mplfinance"]:
    try:
        __import__(pkg)
        print(f"  {pkg}: OK (선택)")
    except ImportError:
        print(f"  {pkg}: 미설치 (Phase A용 선택)")

## 2. Phase 단위 테스트 (Phase B~E, LLM 불필요)

In [ ]:
!python -m pytest tests/test_phase_b.py tests/test_phase_c.py tests/test_phase_d.py tests/test_phase_e.py -v --tb=short

## 3. 실데이터 로드 + 구조 확인

In [ ]:
import pandas as pd

DATA_PATH = r"D:\AutoTrade\Raw_Data\labeled_signals\signals_all_labeled.csv"
assert os.path.exists(DATA_PATH), f"데이터 없음: {DATA_PATH}"

# 샘플 먼저 (1만 행)
df_sample = pd.read_csv(DATA_PATH, nrows=10000, parse_dates=["timestamp"])
print(f"샘플 {len(df_sample):,}행, {len(df_sample.columns)}컬럼")
print("컬럼:", df_sample.columns.tolist())
df_sample.head(3)

In [ ]:
# 필수 컬럼 검증
required = ["symbol", "timestamp", "direction", "confidence", "adx",
            "gap", "exit_type", "realized_roe", "hold_candles",
            "direction_correct"]
missing = [c for c in required if c not in df_sample.columns]
print(f"누락 컬럼: {missing if missing else 'NONE — OK'}")

# exit_type 분포
print("\nexit_type 분포:")
print(df_sample["exit_type"].value_counts())

# 전체 로드 (메모리 여유 있을 때만)
print("\n전체 로드 시작...")
df_all = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])
print(f"전체 {len(df_all):,}행, {df_all['timestamp'].min()} ~ {df_all['timestamp'].max()}")

## 4. 실데이터 통합 — 리스크 파이프라인 소표본 (LLM 호출)

⚠️ **주의**: LLM API 호출이 시그널 수만큼 발생. 여기서는 100건으로 제한.

In [ ]:
from agents.v9_hook import AgentHook
from agents.core.base import reset_llm_usage, get_llm_usage

reset_llm_usage()

hook = AgentHook(
    cg_client=None, btc_trend=None, tg=None, exchange=None,
    enable_risk_pipeline=True,
    enable_memory=True,
)

# 프로덕션 레벨 시그널 100개
sample = df_all.query("confidence >= 60 and adx >= 30 and abs(gap) <= 0.5").head(100)
print(f"샘플: {len(sample)}건")

results = []
for _, row in sample.iterrows():
    signal = {
        "symbol": row["symbol"],
        "direction": row["direction"],
        "confidence": row["confidence"],
        "gap_pct": abs(row["gap"]),
        "adx": row["adx"],
    }
    market_data = {
        "btc_price": float(row.get("btc_price", 90000)),
        "btc_level": int(row.get("btc_level", 2)),
        "btc_atr_pct": float(row.get("atr_pct", 1.5)),
        "fear_greed": int(row.get("fear_greed", 50)),
        "funding_rate": float(row.get("funding_rate", 0.01)),
    }
    result = hook.risk_check(
        signal=signal, positions=[], market_data=market_data,
        account_balance=1000, recent_losses=0,
    )
    if result:
        results.append({
            "symbol": row["symbol"],
            "risk": result["risk_level"],
            "approved": result["approved"],
            "leverage": result["recommended_leverage"],
            "actual_exit": row["exit_type"],
            "actual_roe": row["realized_roe"],
        })

result_df = pd.DataFrame(results)
print(f"\n승인율: {result_df['approved'].mean():.2%}")
print(f"평균 레버리지: {result_df['leverage'].mean():.1f}x")

# 리스크 판단 정확도: 거부된 것들이 실제로 SL히트 많았는가?
rejected = result_df[~result_df["approved"]]
if len(rejected) > 0:
    sl_hit_in_rejected = (rejected["actual_exit"] == "SL").mean()
    print(f"거부 시그널 실제 SL 히트율: {sl_hit_in_rejected:.2%}")

usage = get_llm_usage()
print(f"\n── LLM 사용량 ──")
print(f"  호출: {usage['calls']}")
print(f"  비용: ${usage['estimated_cost_usd']:.4f}")
print(f"  캐시 읽기: {usage['cache_read_tokens']} 토큰")
print(f"  에러: {usage['errors']}")

## 5. Walk-Forward (Baseline + Enhanced, LLM 불필요)

리스크 파이프라인은 LLM 없이 규칙 기반 폴백으로 동작 → 비용 0.

In [ ]:
import subprocess, sys, multiprocessing
WORKERS = min(24, multiprocessing.cpu_count() or 1)
print(f"병렬 워커: {WORKERS}")

cmd = [
    sys.executable, "backtests/walk_forward.py",
    "--data", DATA_PATH,
    "--level", "production",
    "--output", "notebooks/wf_baseline_enhanced.md",
    "--mode", "baseline,enhanced",
    "--workers", str(WORKERS),
]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, encoding="utf-8")
for line in proc.stdout:
    print(line, end="")
proc.wait()
print(f"\n종료 코드: {proc.returncode}")

In [ ]:
with open("notebooks/wf_baseline_enhanced.md", encoding="utf-8") as f:
    print(f.read())

## 6. Walk-Forward (Memory Enhanced, LLM 호출)

⚠️ **비용 주의**: memory 모드는 시그널당 LLM 호출 발생 가능. 윈도우당 2000건으로 제한.

In [ ]:
from agents.core.base import reset_llm_usage, get_llm_usage
reset_llm_usage()

cmd = [
    sys.executable, "backtests/walk_forward.py",
    "--data", DATA_PATH,
    "--level", "production",
    "--output", "notebooks/wf_memory.md",
    "--mode", "baseline,enhanced,memory",
    "--max-signals-per-window", "2000",
    "--workers", str(WORKERS),
]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, encoding="utf-8")
for line in proc.stdout:
    print(line, end="")
proc.wait()
print(f"\n종료 코드: {proc.returncode}")

In [ ]:
with open("notebooks/wf_memory.md", encoding="utf-8") as f:
    print(f.read())

# 누적 LLM 비용 (subprocess 내 호출은 별도 프로세스 → 이 값은 현재 프로세스만)
print("\n현재 프로세스 LLM 사용량 (memory 모드는 subprocess라 여기선 0):")
print(get_llm_usage())

## 7. 레벨별 비교 (선택)

production → level1 → level2 → level3 순으로 필터를 느슨하게.

In [ ]:
for level in ["level1", "level2", "level3"]:
    print(f"\n{'='*60}\n{level}\n{'='*60}")
    subprocess.run([
        sys.executable, "backtests/walk_forward.py",
        "--data", DATA_PATH,
        "--level", level,
        "--output", f"notebooks/wf_{level}.md",
        "--mode", "baseline,enhanced",
        "--workers", str(WORKERS),
    ])

## 8. 결과 통합 분석

In [ ]:
import re

# 각 리포트에서 PF 추출
reports = {
    "production": "notebooks/wf_memory.md",
    "level1": "notebooks/wf_level1.md",
    "level2": "notebooks/wf_level2.md",
    "level3": "notebooks/wf_level3.md",
}

summary = []
for level, path in reports.items():
    if not os.path.exists(path):
        continue
    with open(path, encoding="utf-8") as f:
        text = f.read()
    # PF 변화 요약 섹션 파싱
    for line in text.split("\n"):
        if line.startswith("| Window"):
            parts = [p.strip() for p in line.split("|")[1:-1]]
            if len(parts) >= 5:
                try:
                    summary.append({
                        "level": level,
                        "window": parts[0],
                        "baseline_pf": float(parts[1]),
                        "enhanced_pf": float(parts[2]),
                        "memory_pf": float(parts[3]),
                    })
                except ValueError:
                    pass

if summary:
    summary_df = pd.DataFrame(summary)
    print(summary_df.to_string(index=False))
else:
    print("요약 데이터 없음 — 리포트 먼저 생성 필요")